# DMP Bridge — Run Pipeline

1. **Configure** — set your PDF, model, and extractor
2. **Run** — one call does everything
3. **Result** — the structured DMP output

No annotation rules are applied — this is the raw pipeline output.

## 1 — Configuration

In [1]:
import os
from pathlib import Path

import dmpbridge

# Work from the project root regardless of where Jupyter started
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

# ── Edit these parameters to your needs ───────────────────────────────────────────────
PDF_PATH  = Path("data/input/pdfs/sample10.pdf")
MODEL     = "gemma4:e4b"              # "llama3.1:8b"  "llama3.3:70b"  "gemma4:e4b"
EXTRACTOR = "pdfplumber"              # "pdfplumber"  "docling"  "lightonOCR"
HOST      = "http://localhost:11434"

OUT_DIR         = Path("data/output/pipeline_run")
STRUCTURED_JSON = OUT_DIR / f"{PDF_PATH.stem}_structured.json"

print(f"PDF       : {PDF_PATH}  {'found' if PDF_PATH.exists() else 'NOT FOUND'}")
print(f"Model     : {MODEL}")
print(f"Extractor : {EXTRACTOR}")
print(f"Output    : {STRUCTURED_JSON}")

PDF       : data\input\pdfs\sample10.pdf  found
Model     : gemma4:e4b
Extractor : pdfplumber
Output    : data\output\pipeline_run\sample10_structured.json


## 2 — Run

In [2]:
blocks = dmpbridge.process_pdf(
    PDF_PATH,
    model=MODEL,
    host=HOST,
    extractor=EXTRACTOR,
    structured_output=STRUCTURED_JSON,
    raw_dir=None,
)

print(f"Done — {len(blocks)} blocks labeled")

Done — 22 blocks labeled


## 3 — Structured output Befor applying rules

In [3]:
import json
from IPython.display import HTML

structured = json.loads(STRUCTURED_JSON.read_text(encoding="utf-8"))
template   = structured["narrative"]["template"]

FONT = "font-family:'Inter','Segoe UI',system-ui,Arial,sans-serif;"
print(json.dumps(structured, indent=2, ensure_ascii=False))


def esc(s):
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")

# Document title
doc_title  = template.get("title", "").strip()
title_html = (
    f'<div style="{FONT}background:#0f172a;color:#f8fafc;padding:18px 22px;border-radius:10px;'
    f'margin-bottom:18px;font-size:20px;font-weight:800;">{esc(doc_title)}</div>'
    if doc_title else
    f'<div style="{FONT}background:#e2e8f0;color:#94a3b8;padding:14px 18px;border-radius:10px;'
    f'margin-bottom:18px;font-size:14px;font-style:italic;">No document title detected</div>'
)

# Stats
n_sections  = len(template.get("section", []))
n_questions = sum(len(s.get("question", [])) for s in template.get("section", []))
stats_html  = (
    f'<div style="{FONT}margin-bottom:18px;padding:10px 16px;background:#dbeafe;border-radius:8px;'
    f'font-size:14px;color:#1e40af;font-weight:600;">'
    f'<b>{n_sections}</b> sections &nbsp;·&nbsp; <b>{n_questions}</b> questions'
    f'&nbsp;&nbsp;<span style="color:#64748b;font-weight:400;">|&nbsp; {MODEL} &nbsp;· {EXTRACTOR}</span>'
    f'</div>'
)

# Sections
sections_html = ""
for sec in template.get("section", []):
    sec_title = esc(sec.get("title", "").strip())
    desc      = sec.get("description", "").strip()

    desc_html = (
        f'<div style="{FONT}margin:10px 0 14px 0;padding:10px 14px;background:#f5f3ff;'
        f'border-left:4px solid #6d28d9;border-radius:0 8px 8px 0;'
        f'font-size:14px;color:#4c1d95;font-style:italic;line-height:1.6;">{esc(desc)}</div>'
        if desc else ""
    )

    questions_html = ""
    for q in sec.get("question", []):
        q_text = esc(q.get("text", "").strip())
        ans    = q.get("answer", {}).get("json", {}).get("answer", "").strip()
        ans_html = (
            f'<div style="{FONT}margin-top:8px;padding:10px 14px;background:#f0fdf4;'
            f'border-left:4px solid #059669;border-radius:0 8px 8px 0;'
            f'font-size:15px;color:#1e293b;line-height:1.6;">{esc(ans)}</div>'
            if ans else
            f'<div style="{FONT}margin-top:6px;font-size:13px;color:#94a3b8;font-style:italic;">No answer captured</div>'
        )
        q_label = (
            f'<span style="{FONT}display:inline-block;background:#047857;color:#fff;'
            f'padding:3px 10px;border-radius:10px;font-size:12px;font-weight:700;margin-bottom:6px;">'
            f'Q{q["order"]}</span>'
        )
        q_body = (
            f'<div style="{FONT}font-size:15px;color:#0f172a;font-weight:600;line-height:1.5;">{q_text}</div>'
            if q_text else
            f'<div style="{FONT}font-size:13px;color:#94a3b8;font-style:italic;">(no question text)</div>'
        )
        questions_html += (
            f'<div style="margin:12px 0;padding:12px 16px;background:#fafafa;'
            f'border:1px solid #e2e8f0;border-radius:8px;">'
            f'{q_label}{q_body}{ans_html}</div>'
        )

    if not questions_html:
        questions_html = f'<div style="{FONT}font-size:13px;color:#94a3b8;font-style:italic;padding:6px 0;">No questions</div>'

    sections_html += (
        f'<div style="margin-bottom:20px;border:2px solid #bfdbfe;border-radius:10px;overflow:hidden;">'
        f'<div style="{FONT}background:#1d4ed8;color:#fff;padding:12px 18px;font-weight:700;font-size:16px;">'
        f'Section {sec["order"]}: {sec_title}</div>'
        f'<div style="padding:14px 18px;background:#fff;">{desc_html}{questions_html}</div>'
        f'</div>'
    )

HTML(f'<div style="{FONT}max-width:920px;">{title_html}{stats_html}{sections_html}</div>')

{
  "narrative": {
    "download_url": "",
    "template": {
      "title": "DATA MANAGEMENT",
      "description": "",
      "version": "v1",
      "section": [
        {
          "title": "1. Policy and Practice",
          "description": "",
          "order": 1,
          "question": [
            {
              "text": "",
              "order": 1,
              "answer": {
                "json": {
                  "type": "textArea",
                  "answer": "We will ensure that the data obtained and this study will be made available to the research community.\nThe Bourns College of Engineering (BCOE) at UCR, in partnership with the UCR Libraries and the California Digital Library, has established a new, custom data management system designed to make results of our research readily and reliably accessible. This system combines two services of the California Digital Library (CDL) for the first time: an eScholarship series, a curated site where all BCOE investigators can sto